In [ ]:
import sys
from setup_paths import paths
sys.path.append(paths["project_dir"])

In [ ]:
import json

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import osmnx as ox
import pandas as pd

from ast import literal_eval
from cmap import Colormap
from copy import deepcopy
from matplotlib.colors import Normalize
from os.path import join
from pathlib import Path
from src.prep_and_plot.data import triangular_lattice_graph, generate_all_to_all_demand, save_data
from src.prep_and_plot.plot_helper import total_len_of_bike_paths

In [ ]:
N = 11
save = "triangular"

poly_folder = paths["polygon_folder"]

rng = np.random.default_rng(seed=12345)

In [ ]:
Path(join(paths["plot_folder"], save)).mkdir(parents=True, exist_ok=True)

load_modes_dict = {
    "CyclistsLengthNormed": "000",
    "NormalizedTravelTimePenalty": "010",
    "TotalTravelTimeEstimation": "020",
    "TotalTravelTimeEstimationPerLength": "030",
    "TotalTravelTimeEstimationCapped": "040",
    "TotalTravelTimeEstimationCappedPerLength": "050",
    "TotalDetour": "060",
    "TotalDetourPerLength": "070",
    "TotalTravelTimeEstimationUncapped": "090",
}
load_modes = list(load_modes_dict.values())

modes_dict = {
    "CyclistsLengthNormed": "000",
    "NormalizedTravelTimePenalty": "010",
    "TotalTravelTimeEstimation": "020",
    "TotalTravelTimeEstimationPerLength": "030",
    "TotalTravelTimeEstimationCapped": "040",
    "TotalTravelTimeEstimationCappedPerLength": "050",
    "TotalDetour": "060",
    "TotalDetourPerLength": "070",
    "RandomOrder": "080",
}
modes_dict_rev = {v: k for k, v in modes_dict.items()}
modes = list(modes_dict.values())
mode_base = modes_dict["TotalDetourPerLength"]

labels = {
    "CyclistsLengthNormed": r"$Q_\mathrm{trips}$",
    "NormalizedTravelTimePenalty": r"$Q_\mathrm{pen}$",
    "TotalTravelTimeEstimation": r"$Q_{\widetilde{\Delta T}}$",
    "TotalTravelTimeEstimationPerLength": r"$q_{\widetilde{\Delta T}}$",
    "TotalTravelTimeEstimationCapped": r"$Q_{\widetilde{\Delta T}, \mathrm{capped}}$",
    "TotalTravelTimeEstimationCappedPerLength": r"$q_{\widetilde{\Delta T}, \mathrm{capped}}$",
    "TotalDetour": r"$Q_{\Delta T}$",
    "TotalDetourPerLength": r"$q_{\Delta T}$",
    "TotalTravelTimeEstimationUncapped": r"$Q_{\widetilde{\Delta T}, \mathrm{uncapped}}$",
    "RandomOrder": "random",
}

"""colors = {
    "CyclistsLengthNormed": "#4363d8",
    "NormalizedTravelTimePenalty": "#000075",
    "TotalTravelTimeEstimation": "#dcbeff",
    "TotalTravelTimeEstimationPerLength": "#911eb4",
    "TotalTravelTimeEstimationCapped": "#e6194B",
    "TotalTravelTimeEstimationCappedPerLength": "#800000",
    "TotalDetour": "#ffc819",
    "TotalDetourPerLength": "#f58231",
    "TotalTravelTimeEstimationUncapped": "#a9a9a9",
    "RandomOrder": "#000000",
}"""
colors = {
    "CyclistsLengthNormed": "#4363d8",
    "NormalizedTravelTimePenalty": "#000075",
    "TotalTravelTimeEstimation": "#800000",
    "TotalTravelTimeEstimationPerLength": "#800000",
    "TotalTravelTimeEstimationCapped": "#f58231",
    "TotalTravelTimeEstimationCappedPerLength": "#f58231",
    "TotalDetour": "#ffe119",
    "TotalDetourPerLength": "#ffe119",
    "TotalTravelTimeEstimationUncapped": "#a9a9a9",
    "RandomOrder": "#000000",
}
colors_gs = {
    "CyclistsLengthNormed": "#666666",
    "NormalizedTravelTimePenalty": "#333333",
    "TotalTravelTimeEstimation": "#dcbeff",
    "TotalTravelTimeEstimationPerLength": "#cccccc",
    "TotalTravelTimeEstimationCapped": "#e6194B",
    "TotalTravelTimeEstimationCappedPerLength": "#800000",
    "TotalDetour": "#ffc819",
    "TotalDetourPerLength": "#999999",
    "RandomOrder": "#000000",
}
line_style = {
    "CyclistsLengthNormed": ":",
    "NormalizedTravelTimePenalty": ":",
    "TotalTravelTimeEstimation": ":",
    "TotalTravelTimeEstimationPerLength": "-",
    "TotalTravelTimeEstimationCapped": ":",
    "TotalTravelTimeEstimationCappedPerLength": "-",
    "TotalDetour": ":",
    "TotalDetourPerLength": "-",
    "TotalTravelTimeEstimationUncapped": ":",
    "RandomOrder": ":",
}

mode_base_cap_inf = load_modes_dict["TotalTravelTimeEstimationCapped"]
mode_comp_cap_inf_layer = load_modes_dict["TotalTravelTimeEstimationUncapped"]

realisations = 10

car_penalty_range = np.linspace(start=1.0, stop=10.0, num=10, endpoint=True)
intersection_penalty_range = np.linspace(start=0.0, stop=1.0, num=11, endpoint=True)

pen_sweep_cm = Colormap('colorbrewer:Paired_11')
car_pen_norm = Normalize(vmin=0, vmax=1)

plot_params = {"dpi": 300, "plot_format": "svg", "figsize": (2.8, 2.2), "figsize_wide": (5.4, 2.2), "figsize_map": (3, 4), "figs_snetwork": (5, 5), "fs_title": 10, "fs_ax_label": 6, "fs_ax_ticks": 6, "fs_cbar_label": 6, "fs_cbar_ticks": 6, "edge_lw_thick": 0.75, "edge_lw_thin": 0.25, "edge_color": "#808080", "node_size": 3}

# Data generation

In [ ]:
g, poly = triangular_lattice_graph(N, 2 * N, bike_highways=[i for i in range(2, N + 2)], shift=True)
demand = generate_all_to_all_demand(g, n_ij=None)

input_folder  = join(paths["input_folder"], save)

save_data(g, demand, poly, save, input_folder, poly_folder)

edge_list = [(u, v) for u, v, d in g.edges(data="ramp") if not d]
for i in range(1, 11):
    random_order_i = deepcopy(edge_list)
    rng.shuffle(random_order_i)
    with open(join(paths["input_folder"], save, f"{save}_{i}_random_order.json"), "w") as fp:
            json.dump(random_order_i, fp)

# Plot

In [ ]:
g_plot = ox.load_graphml(join(paths["input_folder"], save, f"{save}.graphml"))
for n, d in g_plot.nodes(data=True):
    g_plot.nodes[n]["level"] = literal_eval(d["level"])
    if "rampnode" in d.keys():
        g_plot.nodes[n]["rampnode"] = literal_eval(d["rampnode"])

edge2segid = {(u, v): idx for idx, (u, v) in enumerate(g_plot.edges())}
segid2edge = {idx: e for e, idx in edge2segid.items()}

In [ ]:
order_segment = pd.DataFrame()

utility = pd.DataFrame()
bpp = pd.DataFrame()
bpp_single_runs = pd.DataFrame()

for mode in load_modes:
    for n in range(1, realisations+1):
        mode_n = f"{mode}_{n:02d}"

        with open(join(paths["output_folder"], save, f"{save}_{n:02d}_data_{mode}.json"), "r") as f:
            data_mode_n = json.load(f)

        order_segment_i = pd.DataFrame({mode_n: [edge2segid[tuple(e)] for e in reversed(data_mode_n["edited_edges"])]})
        order_segment = pd.concat((order_segment, order_segment_i), axis=1)

        bpp_single_runs_i = pd.DataFrame({mode_n: list(reversed(data_mode_n["bike_path_percentage"]))})
        bpp_single_runs = pd.concat((bpp_single_runs, bpp_single_runs_i), axis=1)

        utility_raw_mode = pd.DataFrame({mode_n: list(reversed(data_mode_n["total_utility"]))})
        utility = pd.concat((utility, utility_raw_mode.replace(to_replace=0.0, value=np.nan).dropna()), axis=1)

    order_segment = pd.concat((order_segment, order_segment.rename(columns={f"{mode}_01": mode})[mode]), axis=1)
    bpp = pd.concat((bpp, bpp_single_runs.rename(columns={f"{mode}_01": mode})[mode]), axis=1)

In [ ]:
ba = pd.DataFrame()
ba_single_runs = pd.DataFrame()
ba_interpol = pd.DataFrame()
ba_diff = pd.DataFrame()

utility_opt = utility.max().max()
utility_base = utility.min().min()

for mode in load_modes:
    for n in range(1, realisations+1):
        ba_single_runs[f"{mode}_{n:02d}"] = (utility_base - utility[f"{mode}_{n:02d}"]) / (utility_base - utility_opt)
        ba_interpol[f"{mode}_{n:02d}"] = np.interp(bpp[mode], bpp_single_runs[f"{mode}_{n:02d}"], ba_single_runs[f"{mode}_{n:02d}"])
    ba[mode] = ba_interpol[[f"{mode}_{n:02d}" for n in range(1, realisations+1)]].mean(axis=1)

for mode in load_modes:
    if mode != mode_base:
        ba_base_interpol = np.interp(bpp[mode], bpp[mode_base], ba[mode_base])
        ba_diff[mode] = (ba[mode]-ba_base_interpol)/ba_base_interpol
ba_diff.loc[0] = np.zeros(ba_diff.shape[1])

In [ ]:
utility_random = pd.DataFrame()
bpp_random = pd.DataFrame()

for n in range(1, realisations+1):
    with open(join(paths["output_folder"], save, f"{save}_{n:02d}_data_080.json"), "r") as f:
            data_mode = json.load(f)
    bpp_random[n] = list(reversed(data_mode["bike_path_percentage"]))

    utility_raw_mode = pd.DataFrame(list(reversed(data_mode["total_utility"])))

    utility_random[n] = utility_raw_mode.replace(to_replace=0.0, value=np.nan).dropna()[0].to_list()

ba_random = pd.DataFrame()
ba_random_interpol = pd.DataFrame()

utility_random_opt = utility_random.max().max()
utility_random_base = utility_random.min().min()

for n in range(1, realisations+1):
    ba_random[n] = (utility_random_base - utility_random[n]) / (utility_random_base - utility_random_opt)
    ba_random_interpol[n] = np.interp(bpp_random[1], bpp_random[n], ba_random[n])
bpp["080"] = bpp_random[1]
ba["080"] = ba_random_interpol.mean(axis=1)

ba_base_interpol = np.interp(bpp["080"], bpp[mode_base], ba[mode_base])
ba_diff["080"] = (ba["080"]-ba_base_interpol)/ba_base_interpol
ba_diff.loc[0] = np.zeros(ba_diff.shape[1])

ba_diff_random = pd.DataFrame()

for mode_label, mode in modes_dict.items():
    if mode != "080":
        ba_random_interpol = np.interp(bpp[mode], bpp["080"], ba["080"])
        ba_diff_random[mode] = (ba[mode] - ba_random_interpol)/ba_random_interpol
ba_diff_random.loc[0] = np.zeros(ba_diff_random.shape[1])

In [ ]:
bpp_layers_base_n = {i: np.zeros((len(bpp[mode_base_cap_inf]), realisations)) for i in set([d for n, d in g_plot.nodes(data="level")])}
bpp_layers_base = {i: np.zeros((len(bpp[mode_base_cap_inf]), realisations)) for i in set([d for n, d in g_plot.nodes(data="level")])}

bpp_layers_comp_n = {i: np.zeros((len(bpp[mode_comp_cap_inf_layer]), realisations)) for i in set([d for n, d in g_plot.nodes(data="level")])}
bpp_layers_comp = {i: np.zeros((len(bpp[mode_comp_cap_inf_layer]), realisations)) for i in set([d for n, d in g_plot.nodes(data="level")])}

for layer in bpp_layers_base_n.keys():
    h = deepcopy(g_plot)
    h.remove_nodes_from([n for n, d in g_plot.nodes(data=True) if not d["level"] == layer])
    total_layer_len = total_len_of_bike_paths(h)
    for n in range(1, realisations+1):
        bike_path_layer_len_base = 0
        bike_path_layer_len_comp = 0
        for idx, segment in enumerate(order_segment[f"{mode_base_cap_inf}_{n:02d}"], start=1):
            edge = segid2edge[segment]
            if edge in list(h.edges()):
                bike_path_layer_len_base += h[edge[0]][edge[1]][0]["length"]
            bpp_layers_base_n[layer][idx, n-1] = bike_path_layer_len_base/total_layer_len
        for idx, segment in enumerate(order_segment[f"{mode_comp_cap_inf_layer}_{n:02d}"], start=1):
            edge = segid2edge[segment]
            if edge in list(h.edges()):
                bike_path_layer_len_comp += h[edge[0]][edge[1]][0]["length"]
            bpp_layers_comp_n[layer][idx, n-1] = bike_path_layer_len_comp/total_layer_len
    for n in range(1, realisations+1):
        bpp_layers_base[layer][:, n-1] = np.interp(bpp[mode_base_cap_inf], bpp_single_runs[f"{mode_base_cap_inf}_{n:02d}"], bpp_layers_base_n[layer][:, n-1])
        bpp_layers_comp[layer][:, n-1] = np.interp(bpp[mode_comp_cap_inf_layer], bpp_single_runs[f"{mode_comp_cap_inf_layer}_{n:02d}"], bpp_layers_comp_n[layer][:, n-1])

In [ ]:
fig_cap_inf_ba, ax_cap_inf_ba = plt.subplots(figsize=plot_params["figsize"], dpi=plot_params["dpi"], sharey=True, sharex=True)
ax_cap_inf_ba_in = ax_cap_inf_ba.inset_axes([0.46, 0.07, 0.5, 0.5])

ax_cap_inf_ba_in.axhline(y=0, xmin=0, xmax=1, color="#808080", ls="--", lw=0.5)

for mode_label in ["TotalTravelTimeEstimation", "TotalTravelTimeEstimationCapped", "TotalTravelTimeEstimationUncapped"]:
    mode = load_modes_dict[mode_label]
    ax_cap_inf_ba.plot(bpp[mode], ba[mode], lw=0.75, color=colors[mode_label], label=labels[mode_label])
    if mode != mode_base_cap_inf:
        ba_base_interpol = np.interp(bpp[mode], bpp[mode_base_cap_inf], ba[mode_base_cap_inf])
        ax_cap_inf_ba_in.plot(bpp[mode], (ba[mode]-ba_base_interpol)/ba_base_interpol, lw=0.5, color=colors[mode_label], label=labels[mode_label])

ax_cap_inf_ba.set_xlabel(r"Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])
ax_cap_inf_ba.set_ylabel(r"Bikeability $B(\lambda)$", fontsize=plot_params["fs_ax_label"])

ax_cap_inf_ba.set_xlim(0, 1)
ax_cap_inf_ba.set_ylim(0, 1)
ax_cap_inf_ba.tick_params(axis="y", labelsize=plot_params["fs_ax_ticks"])
ax_cap_inf_ba.tick_params(axis="x", labelsize=plot_params["fs_ax_ticks"])
ax_cap_inf_ba.set_aspect("equal")

ax_cap_inf_ba.legend(fontsize=plot_params["fs_ax_ticks"], loc="center left", bbox_to_anchor=(0.5, 0.75), frameon=False)

ax_cap_inf_ba_in.set_ylabel(r"$\Delta B(\lambda)$ [%]", fontsize=plot_params["fs_ax_label"])

ax_cap_inf_ba_in.tick_params(axis="x", direction="in", labelsize=plot_params["fs_ax_ticks"]-2, pad=1.5)
ax_cap_inf_ba_in.tick_params(axis="y", direction="in", labelsize=plot_params["fs_ax_ticks"]-2, pad=1.5)

ax_cap_inf_ba_in.set_xlim(0, 1)
ax_cap_inf_ba_in.set_ylim(-0.5, 0.2)
ax_cap_inf_ba_in.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, symbol=None))
ax_cap_inf_ba_in.xaxis.set_major_locator(ticker.MultipleLocator(0.2))

fig_cap_inf_ba.savefig(join(paths["plot_folder"], save, f"{save}_cap_inf_ba.{plot_params['plot_format']}"))
plt.show()

In [ ]:
fig_cap_inf_layers, ax_cap_inf_layers = plt.subplots(ncols=2, figsize=plot_params["figsize_wide"], dpi=plot_params["dpi"], sharey=True, sharex=True)

for layer, bpp_layer_base in bpp_layers_base.items():
    ax_cap_inf_layers[1].plot(bpp[mode_base_cap_inf], bpp_layer_base.mean(axis=1), label=layer)
    ax_cap_inf_layers[0].plot(bpp[mode_comp_cap_inf_layer], bpp_layers_comp[layer].mean(axis=1), label=layer)

ax_cap_inf_layers[0].set_xlabel(r"Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])
ax_cap_inf_layers[1].set_xlabel(r"Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])
ax_cap_inf_layers[0].set_ylabel(r"Norm. len. of bike paths per layer $\lambda_k$", fontsize=plot_params["fs_ax_label"])

ax_cap_inf_layers[0].set_xlim(0.0, 1.0)
ax_cap_inf_layers[0].set_ylim(0.0, 1.0)

ax_cap_inf_layers[0].set_aspect("equal")
ax_cap_inf_layers[1].set_aspect("equal")

ax_cap_inf_layers[0].tick_params(axis="x", labelsize=plot_params["fs_ax_ticks"])
ax_cap_inf_layers[1].tick_params(axis="x", labelsize=plot_params["fs_ax_ticks"])
ax_cap_inf_layers[0].tick_params(axis="y", labelsize=plot_params["fs_ax_ticks"])

ax_cap_inf_layers[1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=plot_params["fs_ax_label"], frameon=False)

fig_cap_inf_layers.savefig(join(paths["plot_folder"], save, f"{save}_cap_inf_layers.{plot_params['plot_format']}"), )
plt.show()

In [ ]:
fig_ba, ax_ba = plt.subplots(figsize=(3, 2.5), dpi=plot_params["dpi"], sharey=True, sharex=True)
ax_ba_in = ax_ba.inset_axes([0.46, 0.07, 0.5, 0.5])

ax_ba_in.axhline(y=0, xmin=0, xmax=1, color="#808080", ls="--", lw=0.5)
for mode_label, mode in modes_dict.items():
    ax_ba.plot(bpp[mode], ba[mode], lw=1, color=colors[mode_label], label=labels[mode_label])

    if mode != mode_base:
        ax_ba_in.plot(bpp[mode], ba_diff[mode], lw=0.5, color=colors[mode_label], label=labels[mode_label])

ax_ba.set_xlabel("Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])
ax_ba.set_ylabel("Bikeability $B(\lambda)$", fontsize=plot_params["fs_ax_label"])

ax_ba.set_xlim(0, 0.5)
ax_ba.set_ylim(0, 1)
ax_ba.tick_params(axis="y", labelsize=plot_params["fs_ax_ticks"])
ax_ba.tick_params(axis="x", labelsize=plot_params["fs_ax_ticks"])

ax_ba.legend(fontsize=plot_params["fs_ax_label"], loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)


ax_ba_in.set_ylabel("$\Delta B(\lambda)$ [%]", fontsize=plot_params["fs_ax_label"])

ax_ba_in.tick_params(axis="x", direction="in", labelsize=plot_params["fs_ax_ticks"]-2, pad=1.5)
ax_ba_in.tick_params(axis="y", direction="in", labelsize=plot_params["fs_ax_ticks"]-2, pad=1.5)

ax_ba_in.set_xlim(0, 0.5)
ax_ba_in.set_ylim(-0.3, 0.05)
ax_ba_in.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, symbol=None))
ax_ba_in.xaxis.set_major_locator(ticker.MultipleLocator(0.1))

fig_ba.savefig(join(paths["plot_folder"], save, f"{save}_ba.{plot_params['plot_format']}"))
plt.show()

In [ ]:
ba_diff_area_random_steps = pd.DataFrame()

for mode_label, mode in modes_dict.items():
    if mode != "080":
        ba_random_interpol = np.interp(bpp[mode], bpp["080"], ba["080"])
        ba_diff_area_random_steps[mode] = [np.trapezoid(ba[mode][:idx] - ba_random_interpol[:idx], bpp[mode][:idx]) for idx in range(bpp.shape[0])]

ba_diff_area_random = pd.DataFrame(data={"absolute": ba_diff_area_random_steps.iloc[-1], "relative": 100*ba_diff_area_random_steps.iloc[-1]/ba_diff_area_random_steps.iloc[-1].max()})
ba_diff_area_random.round({"absolute": 3, "relative": 0})

In [ ]:
bpp_layers = {layer: np.load(join(paths["plot_folder"], save, f"{save}_bpp_layers_{layer:02d}_data.npy")) for layer in set([d for n, d in g_plot.nodes(data="level")])}
nbr_ccs = {layer: np.load(join(paths["plot_folder"], save, f"{save}_ccs_layers_{layer:02d}_data.npy")) for layer in set([d for n, d in g_plot.nodes(data="level")])}

In [ ]:
fig_layers, axs_layers = plt.subplots(nrows=2, ncols=5, figsize=(7.4, 3.2), dpi=plot_params["dpi"], sharey=True, sharex=True)
for layer, ax_layers in enumerate(fig_layers.axes):
    for mode_label, mode in {k: v for k, v in modes_dict.items() if v != "080"}.items():
        mode_idx = int(mode[1])
        ax_layers.plot(bpp[mode], bpp_layers[layer][:, mode_idx], label=labels[mode_label], color=colors[mode_label])

    ax_layers.set_xlim(0.0, 1.0)
    ax_layers.set_ylim(0.0, 1.0)

    ax_layers.set_aspect("equal")

    ax_layers.tick_params(axis="x", labelsize=plot_params["fs_ax_ticks"])
    ax_layers.xaxis.set_major_locator(ticker.MultipleLocator(0.2))
    ax_layers.tick_params(axis="y", labelsize=plot_params["fs_ax_ticks"])

    ax_layers.set_title(f"{layer}-layer", fontsize=plot_params["fs_ax_label"], pad=2)

axs_layers[0, 0].set_ylabel("Norm. len. of bike paths \n per layer $\lambda_k$", fontsize=plot_params["fs_ax_label"])
axs_layers[1, 0].set_ylabel("Norm. len. of bike paths \n per layer $\lambda_k$", fontsize=plot_params["fs_ax_label"])
axs_layers[1, 2].set_xlabel("Total normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])
axs_layers[1, 2].legend(loc="lower center", bbox_to_anchor=(0.5, -0.6), ncols=len(modes_dict.keys())-1, fontsize=plot_params["fs_ax_label"], frameon=False)

fig_layers.savefig(join(paths["plot_folder"], save, f"{save}_bpp_layers.{plot_params['plot_format']}"))
plt.show()

In [ ]:
fig_ccs, axs_ccs = plt.subplots(nrows=2, ncols=5, figsize=(7.4, 3.2), dpi=plot_params["dpi"], sharex=True)
for layer, ax_ccs in enumerate(fig_ccs.axes):
    for mode_label, mode in {k: v for k, v in modes_dict.items() if v != "080"}.items():
        mode_idx = int(mode[1])
        ax_ccs.plot(bpp_layers[layer][:, mode_idx], nbr_ccs[layer][:, mode_idx], label=labels[mode_label], color=colors[mode_label])

    ax_ccs.set_xlim(0.0, 1.0)

    ax_ccs.tick_params(axis="x", labelsize=plot_params["fs_ax_ticks"])
    ax_ccs.xaxis.set_major_locator(ticker.MultipleLocator(0.2))
    ax_ccs.tick_params(axis="y", labelsize=plot_params["fs_ax_ticks"], pad=1.5)

    ax_ccs.set_title(f"{layer}-layer", fontsize=plot_params["fs_ax_label"], pad=2)

axs_ccs[0, 0].set_ylabel("Number of strongly \n connected components", fontsize=plot_params["fs_ax_label"])
axs_ccs[1, 0].set_ylabel("Number of strongly \n connected components", fontsize=plot_params["fs_ax_label"])
axs_ccs[1, 2].set_xlabel("Normalised length of bike paths per layer $\lambda_k$", fontsize=plot_params["fs_ax_label"])
axs_ccs[1, 2].legend(loc="lower center", bbox_to_anchor=(0.5, -0.5), ncols=len(modes_dict.keys())-1, fontsize=plot_params["fs_ax_label"], frameon=False)

fig_ccs.savefig(join(paths["plot_folder"], save, f"{save}_ccs_layers.{plot_params['plot_format']}"))
plt.show()

## Car to intersection penalty ratio

In [ ]:
pen_comp = "3-000_0-700"
pen_comp_modes = ["010", "070"]#"000", "030", "020", "040", "050", "060"]

In [ ]:
order_segment_pen_comp = pd.DataFrame()

bpp_pen_comp = pd.DataFrame()
bpp_pen_comp_single_runs = pd.DataFrame()

utility_pen_comp_single_runs = pd.DataFrame()

for mode in [v for v in modes_dict.values() if v != "080"]:
    for n in range(1, realisations+1):
        mode_n = f"{mode}_{n:02d}"

        with open(join(paths["output_folder"], f"{save}_pen_sweep", f"{save}_{pen_comp}_{n:02d}_data_{mode}.json"), "r") as f:
            data_pen_comp_mode_n = json.load(f)

        order_segment_pen_comp_i = pd.DataFrame({mode_n: [edge2segid[tuple(e)] for e in reversed(data_mode_n["edited_edges"])]})
        order_segment_pen_comp = pd.concat((order_segment_pen_comp, order_segment_pen_comp_i), axis=1)

        bpp_pen_comp_single_runs_i = pd.DataFrame({mode_n: list(reversed(data_pen_comp_mode_n["bike_path_percentage"]))})
        bpp_pen_comp_single_runs = pd.concat((bpp_pen_comp_single_runs, bpp_pen_comp_single_runs_i), axis=1)

        utility_pen_comp_raw_mode = pd.DataFrame({mode_n: list(reversed(data_pen_comp_mode_n["total_utility"]))})
        utility_pen_comp_single_runs = pd.concat((utility_pen_comp_single_runs, utility_pen_comp_raw_mode.replace(to_replace=0.0, value=np.nan).dropna()), axis=1)

    order_segment_pen_comp = pd.concat((order_segment_pen_comp, order_segment_pen_comp.rename(columns={f"{mode}_01": mode})[mode]), axis=1)
    bpp_pen_comp = pd.concat((bpp_pen_comp, bpp_single_runs.rename(columns={f"{mode}_01": mode})[mode]), axis=1)

In [ ]:
ba_pen_comp = pd.DataFrame()
ba_pen_comp_single_runs = pd.DataFrame()
ba_pen_comp_interpol = pd.DataFrame()
ba_pen_comp_diff = pd.DataFrame()

utility_pen_comp_opt = utility_pen_comp_single_runs.max().max()
utility_pen_comp_base = utility_pen_comp_single_runs.min().min()

for mode in pen_comp_modes:
    for n in range(1, realisations + 1):
        ba_pen_comp_single_runs[f"{mode}_{n:02d}"] = (utility_pen_comp_base - utility_pen_comp_single_runs[f"{mode}_{n:02d}"]) / (utility_pen_comp_base - utility_pen_comp_opt)
        ba_pen_comp_interpol[f"{mode}_{n:02d}"] = np.interp(bpp_pen_comp[mode], bpp_pen_comp_single_runs[f"{mode}_{n:02d}"], ba_pen_comp_single_runs[f"{mode}_{n:02d}"])
    ba_pen_comp[mode] = ba_pen_comp_interpol[[f"{mode}_{n:02d}" for n in range(1, realisations + 1)]].mean(axis=1)

for mode in pen_comp_modes:
    if mode != mode_base:
        ba_pen_comp_base_interpol = np.interp(bpp_pen_comp[mode], bpp_pen_comp[mode_base], ba_pen_comp[mode_base])
        ba_pen_comp_diff[mode] = (ba_pen_comp[mode] - ba_pen_comp_base_interpol) / ba_pen_comp_base_interpol
ba_pen_comp_diff.loc[0] = np.zeros(ba_pen_comp_diff.shape[1])

In [ ]:
utility_pen_comp_random = pd.DataFrame()
bpp_pen_comp_random = pd.DataFrame()

for n in range(1, realisations + 1):
    with open(join(paths["output_folder"], f"{save}_pen_sweep", f"{save}_{pen_comp}_{n:02d}_data_080.json"), "r") as f:
        data_mode = json.load(f)
    bpp_pen_comp_random[n] = list(reversed(data_mode["bike_path_percentage"]))

    utility_pen_comp_raw_mode = pd.DataFrame(list(reversed(data_mode["total_utility"])))

    utility_pen_comp_random[n] = utility_pen_comp_raw_mode.replace(to_replace=0.0, value=np.nan).dropna()[0].to_list()

ba_pen_comp_random = pd.DataFrame()
ba_pen_comp_random_interpol = pd.DataFrame()

utility_pen_comp_random_opt = utility_pen_comp_random.max().max()
utility_pen_comp_random_base = utility_pen_comp_random.min().min()

for n in range(1, realisations + 1):
    ba_pen_comp_random[n] = (utility_pen_comp_random_base - utility_pen_comp_random[n]) / (utility_pen_comp_random_base - utility_pen_comp_random_opt)
    ba_pen_comp_random_interpol[n] = np.interp(bpp_pen_comp_random[1], bpp_pen_comp_random[n], ba_pen_comp_random[n])
bpp_pen_comp["080"] = bpp_pen_comp_random[1]
ba_pen_comp["080"] = ba_pen_comp_random_interpol.mean(axis=1)

ba_pen_comp_base_interpol = np.interp(bpp_pen_comp["080"], bpp_pen_comp[mode_base], ba_pen_comp[mode_base])
ba_pen_comp_diff["080"] = (ba_pen_comp["080"] - ba_pen_comp_base_interpol) / ba_pen_comp_base_interpol
ba_pen_comp_diff.loc[0] = np.zeros(ba_pen_comp_diff.shape[1])

In [ ]:
bpp_layers_pen_comp = {i: np.zeros(bpp_pen_comp.shape) for i in set([d for n, d in g_plot.nodes(data="level")])}

for layer in bpp_layers_pen_comp.keys():
    h = deepcopy(g_plot)
    h.remove_nodes_from([n for n, d in g_plot.nodes(data=True) if not d["level"] == layer])
    h.remove_edges_from([(u, v, k) for u, v, k, d in g_plot.edges(keys=True, data=True) if d["length"] < 1.0])
    total_layer_len = total_len_of_bike_paths(h)
    for mode in pen_comp_modes:
        mode_idx = int(mode[1])
        bpp_layers_pen_comp_n = np.zeros((bpp.shape[0], realisations))
        nbr_ccs_n = np.zeros((bpp.shape[0], realisations))
        for n in range(1, realisations+1):
            nbr_ccs_layer = 0
            bike_path_layer_len = 0
            bike_paths_layer = []
            for idx, segment in enumerate(order_segment[f"{mode}_{n:02d}"], start=1):
                edge = segid2edge[segment]
                if edge in list(h.edges()):
                    bike_path_layer_len += h[edge[0]][edge[1]][0]["length"]

                bpp_layers_pen_comp_n[idx, n-1] = bike_path_layer_len/total_layer_len

        bpp_layers_pen_comp_interpol = np.zeros((bpp.shape[0], realisations))
        for n in range(1, realisations+1):
            bpp_layers_pen_comp_interpol[:, n-1] = np.interp(bpp_pen_comp[mode], bpp_pen_comp_single_runs[f"{mode}_{n:02d}"], bpp_layers_pen_comp_n[:, n-1])
        bpp_layers_pen_comp[layer][:, mode_idx] = bpp_layers_pen_comp_interpol.mean(axis=1)

In [ ]:
fig_pen_comp, axs_pen_comp = plt.subplots(ncols=4, figsize=(7.4, 2.2), dpi=plot_params["dpi"], sharey=True, sharex=True)
axs_pen_comp_ba_in = axs_pen_comp[0].inset_axes([0.46, 0.07, 0.5, 0.5])
axs_pen_comp_ba_in.axhline(y=0, xmin=0, xmax=1, color="#808080", ls="--", lw=0.5)

for mode_label, mode in {k: v for k, v in modes_dict.items() if v in pen_comp_modes+["080"]}.items():
    axs_pen_comp[0].plot(bpp_pen_comp[mode], ba_pen_comp[mode], lw=0.75, color=colors[mode_label], label=labels[mode_label])
    axs_pen_comp[0].plot(bpp[mode], ba[mode], lw=0.75, ls="--", color=colors_gs[mode_label], label=labels[mode_label], alpha=0.75)

    axs_pen_comp[0].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"])
    axs_pen_comp[0].xaxis.set_major_locator(ticker.MultipleLocator(0.2))
    axs_pen_comp[0].set_xlim(0.0, 1)
    axs_pen_comp[0].set_ylim(0, 1)
    axs_pen_comp[0].set_aspect("equal")

    if mode != mode_base:
        axs_pen_comp_ba_in.plot(bpp[mode], ba_diff[mode], lw=0.5, ls="--", color=colors_gs[mode_label], label=labels[mode_label], alpha=0.75)
        axs_pen_comp_ba_in.plot(bpp_pen_comp[mode], ba_pen_comp_diff[mode], lw=0.5, color=colors[mode_label], label=labels[mode_label])

    if mode != "080":
        mode_idx = int(mode[1])
        for layer_idx, layer in enumerate([0, 2, 5], start=1):
            axs_pen_comp[layer_idx].plot(bpp_pen_comp[mode], bpp_layers_pen_comp[layer][:, mode_idx], color=colors[mode_label], label=labels[mode_label])
            axs_pen_comp[layer_idx].plot(bpp[mode], bpp_layers[layer][:, mode_idx], lw=0.75, ls="--", color=colors_gs[mode_label], label=labels[mode_label], alpha=0.75)

            axs_pen_comp[layer_idx].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"])
            axs_pen_comp[layer_idx].xaxis.set_major_locator(ticker.MultipleLocator(0.2))
            axs_pen_comp[layer_idx].set_xlim(0, 1)
            axs_pen_comp[layer_idx].set_ylim(0, 1)
            axs_pen_comp[layer_idx].set_aspect("equal")

            axs_pen_comp[layer_idx].set_title(f"{layer}-layer", fontsize=plot_params["fs_ax_label"], pad=2.25)


axs_pen_comp[0].set_xlabel("Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])
axs_pen_comp[2].set_xlabel("Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])

axs_pen_comp[0].set_ylabel("Bikeability $B(\lambda)$", fontsize=plot_params["fs_ax_label"])
axs_pen_comp[1].set_ylabel("Norm. len. of bike paths per layer $\lambda_k$", fontsize=plot_params["fs_ax_label"])

axs_pen_comp[0].legend(loc="lower left", bbox_to_anchor=(0.0, 0.0), ncols=1, fontsize=plot_params["fs_ax_ticks"], frameon=False)
axs_pen_comp[1].legend(loc="lower center", bbox_to_anchor=(0.55, 0.0), ncols=len(pen_comp_modes), fontsize=plot_params["fs_ax_ticks"], frameon=False)

axs_pen_comp_ba_in.set_ylabel("$\Delta B(\lambda)$ [%]", fontsize=plot_params["fs_ax_label"], labelpad=1.5)
axs_pen_comp_ba_in.tick_params(axis="x", direction="in", labelsize=plot_params["fs_ax_ticks"]-2, pad=1.5)
axs_pen_comp_ba_in.tick_params(axis="y", direction="in", labelsize=plot_params["fs_ax_ticks"]-2, pad=1.5)
axs_pen_comp_ba_in.set_xlim(0, 0.5)
axs_pen_comp_ba_in.set_ylim(-0.15, 0.0)
axs_pen_comp_ba_in.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, symbol=None))
axs_pen_comp_ba_in.xaxis.set_major_locator(ticker.MultipleLocator(0.1))

fig_pen_comp.savefig(join(paths["plot_folder"], save, f"{save}_pen_comp.{plot_params['plot_format']}"))
plt.show()

In [ ]:
ba_pen_comp_diff_area_random_steps = pd.DataFrame()

for mode in pen_comp_modes:
    ba_pen_comp_random_interpol = np.interp(bpp_pen_comp[mode], bpp_pen_comp["080"], ba_pen_comp["080"])
    ba_pen_comp_diff_area_random_steps[mode] = [np.trapezoid(ba_pen_comp[mode][:idx] - ba_pen_comp_random_interpol[:idx], bpp_pen_comp[mode][:idx]) for idx in range(bpp_pen_comp.shape[0])]

ba_pen_comp_diff_area_random = pd.DataFrame(data={"absolute": ba_pen_comp_diff_area_random_steps.iloc[-1], "relative": 100*ba_pen_comp_diff_area_random_steps.iloc[-1]/ba_pen_comp_diff_area_random_steps.iloc[-1].max()})
ba_pen_comp_diff_area_random.round({"absolute": 3, "relative": 0})

## Pen Sweep

In [ ]:
mode_car_inter_pen = load_modes_dict["NormalizedTravelTimePenalty"]
car_inter_pen_plot_combinations = {"1-000_1-000": ("1.0", 1.0), "2-000_0-800": ("1/2", 0.8), "3-000_0-700": ("1/3", 0.7), "4-000_0-600": ("1/4", 0.6), "5-000_0-500": ("1/5", 0.5), "6-000_0-400": ("1/6", 0.4), "7-000_0-300": ("1/7", 0.3), "8-000_0-200": ("1/8", 0.2), "9-000_0-100": ("1/9", 0.1), "10-000_0-000": ("1/10", 0.0)}

In [ ]:
order_segment_car_inter_pen = pd.DataFrame()
bpp_car_inter_pen = pd.DataFrame()
bpp_car_inter_pen_random = pd.DataFrame()

car_inter_pen_2_idx = dict()

utility_car_inter_pen = pd.DataFrame()
utility_car_inter_pen_random = pd.DataFrame()

car_inter_pen_idx = 0
for c_pen in car_penalty_range:
    for i_pen in intersection_penalty_range:
        car_inter_pen = f"{c_pen:2.3f}_{i_pen:2.3f}".replace(".", "-")
        car_inter_pen_2_idx[car_inter_pen] = car_inter_pen_idx
        car_inter_pen_idx += 1

        for n in range(1, realisations+1):
            car_inter_pen_n = f"{car_inter_pen}_{n:02d}"
            save_car_inter_pen_n = f"{save}_{car_inter_pen_n}"
            save_folder_car_inter_pen = f"{save}_pen_sweep"

            with open(join(paths["output_folder"], save_folder_car_inter_pen, f"{save_car_inter_pen_n}_data_{mode_car_inter_pen}.json"), "r") as f:
                data_car_inter_pen = json.load(f)
            with open(join(paths["output_folder"], save_folder_car_inter_pen, f"{save_car_inter_pen_n}_data_080.json"), "r") as f:
                data_car_inter_pen_random = json.load(f)

            order_segment_car_inter_pen_i = pd.DataFrame({car_inter_pen_n: [edge2segid[tuple(e)] for e in reversed(data_car_inter_pen["edited_edges"])]})
            order_segment_car_inter_pen = pd.concat((order_segment_car_inter_pen, order_segment_car_inter_pen_i), axis=1)

            bpp_car_inter_pen_i = pd.DataFrame({car_inter_pen_n: list(reversed(data_car_inter_pen["bike_path_percentage"]))})
            bpp_car_inter_pen = pd.concat((bpp_car_inter_pen, bpp_car_inter_pen_i), axis=1)

            bpp_car_inter_pen_random_i = pd.DataFrame({car_inter_pen_n: list(reversed(data_car_inter_pen_random["bike_path_percentage"]))})
            bpp_car_inter_pen_random = pd.concat((bpp_car_inter_pen_random, bpp_car_inter_pen_random_i), axis=1)

            utility_raw_car_inter_pen = pd.DataFrame({car_inter_pen_n: list(reversed(data_car_inter_pen["total_utility"]))})
            utility_car_inter_pen = pd.concat((utility_car_inter_pen, utility_raw_car_inter_pen.replace(to_replace=0.0, value=np.nan).dropna()), axis=1)
            utility_raw_car_inter_pen_random = pd.DataFrame({car_inter_pen_n: list(reversed(data_car_inter_pen_random["total_utility"]))})
            utility_car_inter_pen_random = pd.concat((utility_car_inter_pen_random, utility_raw_car_inter_pen_random.replace(to_replace=0.0, value=np.nan).dropna()), axis=1)

        bpp_car_inter_pen = pd.concat((bpp_car_inter_pen, bpp_car_inter_pen.rename(columns={f"{car_inter_pen}_01": car_inter_pen})[car_inter_pen]), axis=1)
        bpp_car_inter_pen_random = pd.concat((bpp_car_inter_pen_random, bpp_car_inter_pen_random.rename(columns={f"{car_inter_pen}_01": car_inter_pen})[car_inter_pen]), axis=1)

In [ ]:
ba_car_inter_pen = pd.DataFrame()
ba_car_inter_pen_random = pd.DataFrame()

for c_pen in car_penalty_range:
    for i_pen in intersection_penalty_range:
        car_inter_pen = f"{c_pen:2.3f}_{i_pen:2.3f}".replace(".", "-")

        ba_car_inter_pen_interpol = pd.DataFrame()
        ba_car_inter_pen_random_interpol = pd.DataFrame()
        for n in range(1, realisations+1):
            car_inter_pen_n = f"{car_inter_pen}_{n:02d}"

            utility_car_inter_pen_n_opt = utility_car_inter_pen[car_inter_pen_n].max()
            utility_car_inter_pen_n_base = utility_car_inter_pen[car_inter_pen_n].min()
            utility_car_inter_pen_random_n_opt = utility_car_inter_pen_random[car_inter_pen_n].max()
            utility_car_inter_pen_random_n_base = utility_car_inter_pen_random[car_inter_pen_n].min()

            ba_car_inter_pen_n = (utility_car_inter_pen_n_base - utility_car_inter_pen[car_inter_pen_n]) / (utility_car_inter_pen_n_base - utility_car_inter_pen_n_opt)
            ba_car_inter_pen_interpol[car_inter_pen_n] = np.interp(bpp_car_inter_pen[car_inter_pen], bpp_car_inter_pen[car_inter_pen_n], ba_car_inter_pen_n)

            ba_car_inter_pen_random_n = (utility_car_inter_pen_random_n_base - utility_car_inter_pen_random[car_inter_pen_n]) / (utility_car_inter_pen_random_n_base - utility_car_inter_pen_random_n_opt)
            ba_car_inter_pen_random_interpol[car_inter_pen_n] = np.interp(bpp_car_inter_pen_random[car_inter_pen], bpp_car_inter_pen_random[car_inter_pen_n], ba_car_inter_pen_random_n)

        ba_car_inter_pen_interpol[car_inter_pen] = ba_car_inter_pen_interpol[[f"{car_inter_pen}_{n:02d}" for n in range(1, realisations+1)]].mean(axis=1)
        ba_car_inter_pen = pd.concat((ba_car_inter_pen, ba_car_inter_pen_interpol[car_inter_pen]), axis=1)
        ba_car_inter_pen_random_interpol[car_inter_pen] = ba_car_inter_pen_random_interpol[[f"{car_inter_pen}_{n:02d}" for n in range(1, realisations+1)]].mean(axis=1)
        ba_car_inter_pen_random = pd.concat((ba_car_inter_pen_random, ba_car_inter_pen_random_interpol[car_inter_pen]), axis=1)

In [ ]:
ba_diff_area_random_car_inter_pen = pd.DataFrame()

for c_pen in car_penalty_range:
    for i_pen in intersection_penalty_range:
        car_inter_pen = f"{c_pen:2.3f}_{i_pen:2.3f}".replace(".", "-")
        ba_car_inter_pen_random_interpol = np.interp(bpp_car_inter_pen[car_inter_pen], bpp_car_inter_pen_random[car_inter_pen], ba_car_inter_pen_random[car_inter_pen])
        ba_diff_area_random_car_inter_pen = pd.concat((ba_diff_area_random_car_inter_pen, pd.DataFrame({car_inter_pen: [np.trapezoid(ba_car_inter_pen[car_inter_pen] - ba_car_inter_pen_random_interpol, bpp_car_inter_pen[car_inter_pen])]})), axis=1)

ba_diff_area_random_car_inter_pen["1-000_0-000"] = np.mean([ba_diff_area_random_car_inter_pen[f"1-000_{i_pen:2.3f}".replace(".", "-")][0] for i_pen in intersection_penalty_range[1:]])

In [ ]:
bpp_layers_car_inter_pen = {layer: np.load(join(paths["plot_folder"], save, f"{save}_pen_sweep_bpp_layers_{layer:02d}_data_{mode_car_inter_pen}.npy")) for layer in set([d for n, d in g_plot.nodes(data="level")])}
nbr_ccs_car_inter_pen = {layer: np.load(join(paths["plot_folder"], save, f"{save}_pen_sweep_ccs_layers_{layer:02d}_data_{mode_car_inter_pen}.npy")) for layer in set([d for n, d in g_plot.nodes(data="level")])}

In [ ]:
fig_pen_sweep, axs_pen_sweep = plt.subplots(ncols=4, figsize=(7.4, 2.2), dpi=plot_params["dpi"], sharey=True, sharex=True)

for car_inter_pen, car_inter_pen_values in car_inter_pen_plot_combinations.items():
    car_inter_pen_idx = car_inter_pen_2_idx[car_inter_pen]
    car_inter_pen_label = f"{car_inter_pen_values[0]}, {car_inter_pen_values[1]:2.1f}"

    axs_pen_sweep[0].plot(bpp_car_inter_pen[car_inter_pen], ba_car_inter_pen[car_inter_pen], lw=1, label=car_inter_pen_label)

    axs_pen_sweep[0].set_xlim(0, 1)
    axs_pen_sweep[0].set_ylim(0, 1)
    axs_pen_sweep[0].set_aspect("equal")
    axs_pen_sweep[0].xaxis.set_major_locator(ticker.MultipleLocator(0.2))
    axs_pen_sweep[0].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"])

    for layer_idx, layer in enumerate([0, 2, 5], start=1):
        axs_pen_sweep[layer_idx].set_title(f"{layer}-layer", fontsize=plot_params["fs_ax_label"], pad=2.25)
        axs_pen_sweep[layer_idx].plot(bpp_car_inter_pen[car_inter_pen], bpp_layers_car_inter_pen[layer][:, car_inter_pen_idx], lw=1,label=car_inter_pen_label)

        axs_pen_sweep[layer_idx].set_xlim(0, 1)
        axs_pen_sweep[layer_idx].set_ylim(0, 1)
        axs_pen_sweep[layer_idx].set_aspect("equal")
        axs_pen_sweep[layer_idx].xaxis.set_major_locator(ticker.MultipleLocator(0.2))
        axs_pen_sweep[layer_idx].tick_params(axis="both", labelsize=plot_params["fs_ax_ticks"])

axs_pen_sweep[0].set_xlabel("Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])
axs_pen_sweep[2].set_xlabel("Normalised length of bike paths $\lambda$", fontsize=plot_params["fs_ax_label"])
axs_pen_sweep[0].set_ylabel("Bikeability $B(\lambda)$", fontsize=plot_params["fs_ax_label"])
axs_pen_sweep[1].set_ylabel("Norm. len. of bike paths per layer $\lambda_k$", fontsize=plot_params["fs_ax_label"])

axs_pen_sweep[0].legend(loc="lower center", bbox_to_anchor=(0.6, 0.1), ncols=2, fontsize=plot_params["fs_ax_ticks"], frameon=False)

fig_pen_sweep.savefig(join(paths["plot_folder"], save, f"{save}_pen_sweep.{plot_params['plot_format']}"))
plt.show()